# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aijaz-khalique/flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



I use a simple decision-support rule to rank content items with unusually weak CTR for their search-position context.

The main signal is the difference between observed CTR and the typical CTR for similar search-position contexts.

A more negative CTR gap means the item is receiving fewer clicks than expected.

I also use impression volume as a confidence signal. A negative CTR gap supported by more impressions is more useful for prioritization.

The baseline score gives 80% weight to CTR-gap severity and 20% weight to impression volume.

Reason codes:

- `LOW_CTR_FOR_POSITION` — the item's CTR is below the position-context benchmark.
- `NO_CTR_OPPORTUNITY` — the item does not have a negative CTR gap.

Action labels:

- `REVIEW_CTR_OPPORTUNITY` — manually inspect the title, snippet, search intent, and page presentation.
- `MONITOR` — no immediate review is recommended.

This is a directional decision-support rule, not a guarantee that the content will decline or that changing it will improve rankings.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not set. Please set your Hugging Face token."
    )

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    token=HF_TOKEN
)

df = dataset.to_pandas()

print("Data loaded successfully!")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Data loaded successfully!
Rows: 2414248
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [8]:
queue = df.copy()

# Convert numeric fields
numeric_columns = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d"
]

for col in numeric_columns:
    queue[col] = pd.to_numeric(
        queue[col],
        errors="coerce"
    )

# Keep rows with valid impressions
queue = queue.dropna(
    subset=[
        "impressions_90d",
        "clicks_90d",
        "avg_position_90d"
    ]
).copy()

# Remove zero/negative impressions
queue = queue[
    queue["impressions_90d"] > 0
].copy()

print("Rows available for scoring:", len(queue))

Rows available for scoring: 2414248


In [9]:
queue["ctr_90d"] = (
    queue["clicks_90d"]
    / queue["impressions_90d"]
)

print("Observed CTR created.")

display(
    queue[
        [
            "impressions_90d",
            "clicks_90d",
            "ctr_90d"
        ]
    ].head()
)

Observed CTR created.


,impressions_90d,clicks_90d,ctr_90d
0,11,0,0.0
1,13,0,0.0
2,16,0,0.0
3,55,0,0.0
4,14,0,0.0


In [10]:
queue["position_bucket"] = pd.cut(
    queue["avg_position_90d"],
    bins=[
        0,
        3,
        5,
        10,
        20,
        50,
        np.inf
    ],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "21-50",
        "51+"
    ],
    include_lowest=True
)

print("Position buckets created.")

display(
    queue[
        [
            "avg_position_90d",
            "position_bucket"
        ]
    ].head(10)
)

Position buckets created.


,avg_position_90d,position_bucket
0,10.818182,11-20
1,1.769231,1-3
2,23.562500,21-50
3,2.200000,1-3
4,3.428571,4-5
5,2.775000,1-3
6,17.062500,11-20
7,52.571429,51+
8,88.058824,51+
9,13.636364,11-20


In [11]:
position_benchmark = (
    queue
    .groupby("position_bucket", observed=True)
    .agg(
        expected_ctr=("ctr_90d", "median"),
        benchmark_rows=("ctr_90d", "size")
    )
    .reset_index()
)

print("Position CTR benchmark:")

display(position_benchmark)

Position CTR benchmark:


,position_bucket,expected_ctr,benchmark_rows
0,1-3,0.0,405442
1,4-5,0.0,205617
2,6-10,0.0,772872
3,11-20,0.0,327380
4,21-50,0.0,366561
5,51+,0.0,336376


In [12]:
queue = queue.merge(
    position_benchmark[
        [
            "position_bucket",
            "expected_ctr"
        ]
    ],
    on="position_bucket",
    how="left"
)

print("Expected CTR added.")

display(
    queue[
        [
            "ctr_90d",
            "position_bucket",
            "expected_ctr"
        ]
    ].head()
)

Expected CTR added.


,ctr_90d,position_bucket,expected_ctr
0,0.0,11-20,0.0
1,0.0,1-3,0.0
2,0.0,21-50,0.0
3,0.0,1-3,0.0
4,0.0,4-5,0.0


In [13]:
queue["ctr_gap"] = (
    queue["ctr_90d"]
    - queue["expected_ctr"]
)

print("ctr_gap created successfully.")

display(
    queue[
        [
            "ctr_90d",
            "expected_ctr",
            "ctr_gap"
        ]
    ].head(20)
)

print("\nCTR gap statistics:")

display(
    queue["ctr_gap"].describe()
)

ctr_gap created successfully.


,ctr_90d,expected_ctr,ctr_gap
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0
5,0.0,0.0,0.0
6,0.0,0.0,0.0
7,0.0,0.0,0.0
8,0.0,0.0,0.0
9,0.0,0.0,0.0



CTR gap statistics:


count    2.414248e+06
mean     2.030823e-03
std      1.080019e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.333333e-01
Name: ctr_gap, dtype: float64

In [14]:
queue["ctr_gap_severity"] = (
    -queue["ctr_gap"].clip(upper=0)
)

max_severity = queue[
    "ctr_gap_severity"
].max()

if max_severity > 0:

    queue["ctr_gap_score"] = (
        queue["ctr_gap_severity"]
        / max_severity
        * 100
    )

else:

    queue["ctr_gap_score"] = 0.0

print("CTR-gap severity calculated.")

display(
    queue[
        [
            "ctr_gap",
            "ctr_gap_severity",
            "ctr_gap_score"
        ]
    ].head()
)

CTR-gap severity calculated.


,ctr_gap,ctr_gap_severity,ctr_gap_score
0,0.0,-0.0,0.0
1,0.0,-0.0,0.0
2,0.0,-0.0,0.0
3,0.0,-0.0,0.0
4,0.0,-0.0,0.0


In [15]:
queue["volume_raw"] = np.log1p(
    queue["impressions_90d"]
)

max_volume = queue[
    "volume_raw"
].max()

if max_volume > 0:

    queue["volume_score"] = (
        queue["volume_raw"]
        / max_volume
        * 100
    )

else:

    queue["volume_score"] = 0.0

print("Volume confidence calculated.")

Volume confidence calculated.


In [16]:
queue["baseline_score"] = (
    0.80 * queue["ctr_gap_score"]
    +
    0.20 * queue["volume_score"]
)

print("Baseline score calculated.")

display(
    queue[
        [
            "ctr_gap_score",
            "volume_score",
            "baseline_score"
        ]
    ].head()
)

Baseline score calculated.


,ctr_gap_score,volume_score,baseline_score
0,0.0,18.817997,3.763599
1,0.0,19.985368,3.997074
2,0.0,21.455696,4.291139
3,0.0,30.483663,6.096733
4,0.0,20.507845,4.101569


In [17]:
queue["reason_code"] = np.where(
    (
        (queue["ctr_gap"] < 0)
        &
        (queue["impressions_90d"] > 0)
    ),
    "LOW_CTR_FOR_POSITION",
    "NO_CTR_OPPORTUNITY"
)

print("Reason codes created.")

print(
    queue["reason_code"].value_counts()
)

Reason codes created.
reason_code
NO_CTR_OPPORTUNITY    2414248
Name: count, dtype: int64


In [18]:
queue["action_label"] = np.where(
    queue["reason_code"]
    == "LOW_CTR_FOR_POSITION",

    "REVIEW_CTR_OPPORTUNITY",

    "MONITOR"
)

print("Action labels created.")

print(
    queue["action_label"].value_counts()
)

Action labels created.
action_label
MONITOR    2414248
Name: count, dtype: int64


In [19]:
queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

print("Ranking completed.")

display(
    queue[
        [
            "rank",
            "content_hash_id",
            "impressions_90d",
            "ctr_90d",
            "expected_ctr",
            "ctr_gap",
            "baseline_score",
            "reason_code",
            "action_label"
        ]
    ].head(20)
)

Ranking completed.


,rank,content_hash_id,impressions_90d,ctr_90d,expected_ctr,ctr_gap,baseline_score,reason_code,action_label
0,1,content_943dc881428182b8,543044,0.000101,0.0,0.000101,20.000000,NO_CTR_OPPORTUNITY,MONITOR
1,2,content_11bf4c33adea7bdc,331832,0.000000,0.0,0.000000,19.253976,NO_CTR_OPPORTUNITY,MONITOR
2,3,content_d0acf7062bc6b257,292374,0.000000,0.0,0.000000,19.062238,NO_CTR_OPPORTUNITY,MONITOR
3,4,content_c60628276389acbb,282446,0.000000,0.0,0.000000,19.009915,NO_CTR_OPPORTUNITY,MONITOR
4,5,content_99fc6465edb0e52c,278990,0.000000,0.0,0.000000,18.991268,NO_CTR_OPPORTUNITY,MONITOR
5,6,content_39e19a3ec2d95f9d,264399,0.000000,0.0,0.000000,18.909910,NO_CTR_OPPORTUNITY,MONITOR
6,7,content_987d251ee617d9c6,261191,0.002998,0.0,0.002998,18.891421,NO_CTR_OPPORTUNITY,MONITOR
7,8,content_32c5cc913fb4ff41,256491,0.000008,0.0,0.000008,18.863919,NO_CTR_OPPORTUNITY,MONITOR
8,9,content_012de75c008aa653,248191,0.000000,0.0,0.000000,18.814097,NO_CTR_OPPORTUNITY,MONITOR
9,10,content_7471467133493ce6,209105,0.000000,0.0,0.000000,18.554556,NO_CTR_OPPORTUNITY,MONITOR


In [20]:
output_columns = [
    "rank",
    "content_hash_id",
    "impressions_90d",
    "ctr_90d",
    "expected_ctr",
    "ctr_gap",
    "ctr_gap_score",
    "volume_score",
    "baseline_score",
    "reason_code",
    "action_label"
]

baseline_queue = queue[
    output_columns
].copy()

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_queue.to_csv(
    output_path,
    index=False
)

print("================================")
print("CSV SAVED SUCCESSFULLY")
print("================================")
print(output_path)
print("Rows:", len(baseline_queue))

display(
    baseline_queue.head(20)
)

CSV SAVED SUCCESSFULLY
work\outputs\baseline_action_score.csv
Rows: 2414248


,rank,content_hash_id,impressions_90d,ctr_90d,expected_ctr,ctr_gap,ctr_gap_score,volume_score,baseline_score,reason_code,action_label
0,1,content_943dc881428182b8,543044,0.000101,0.0,0.000101,0.0,100.000000,20.000000,NO_CTR_OPPORTUNITY,MONITOR
1,2,content_11bf4c33adea7bdc,331832,0.000000,0.0,0.000000,0.0,96.269880,19.253976,NO_CTR_OPPORTUNITY,MONITOR
2,3,content_d0acf7062bc6b257,292374,0.000000,0.0,0.000000,0.0,95.311189,19.062238,NO_CTR_OPPORTUNITY,MONITOR
3,4,content_c60628276389acbb,282446,0.000000,0.0,0.000000,0.0,95.049573,19.009915,NO_CTR_OPPORTUNITY,MONITOR
4,5,content_99fc6465edb0e52c,278990,0.000000,0.0,0.000000,0.0,94.956340,18.991268,NO_CTR_OPPORTUNITY,MONITOR
5,6,content_39e19a3ec2d95f9d,264399,0.000000,0.0,0.000000,0.0,94.549550,18.909910,NO_CTR_OPPORTUNITY,MONITOR
6,7,content_987d251ee617d9c6,261191,0.002998,0.0,0.002998,0.0,94.457105,18.891421,NO_CTR_OPPORTUNITY,MONITOR
7,8,content_32c5cc913fb4ff41,256491,0.000008,0.0,0.000008,0.0,94.319593,18.863919,NO_CTR_OPPORTUNITY,MONITOR
8,9,content_012de75c008aa653,248191,0.000000,0.0,0.000000,0.0,94.070483,18.814097,NO_CTR_OPPORTUNITY,MONITOR
9,10,content_7471467133493ce6,209105,0.000000,0.0,0.000000,0.0,92.772778,18.554556,NO_CTR_OPPORTUNITY,MONITOR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



The top 20 items are decision-support candidates, not guaranteed problems.

Items with larger negative CTR gaps receive higher priority. Impression volume provides an additional confidence signal.

The result should be manually reviewed because CTR can be affected by query mix, SERP features, seasonality, and other factors.

The recommended action is review, not automatic modification.

In [21]:
top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["volume_score"] >= 70,
    "Higher confidence: strong relative impression volume.",
    np.where(
        top20["volume_score"] >= 40,
        "Moderate confidence: reasonable impression volume.",
        "Lower confidence: limited impression volume."
    )
)

top20["what_would_make_it_wrong"] = (
    "Query mix, SERP features, position benchmark, "
    "seasonality, or random variation may explain the gap."
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action_label",
        "reason_code",
        "impressions_90d",
        "ctr_90d",
        "expected_ctr",
        "ctr_gap",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_hash_id,action_label,reason_code,impressions_90d,ctr_90d,expected_ctr,ctr_gap,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_943dc881428182b8,MONITOR,NO_CTR_OPPORTUNITY,543044,0.000101,0.0,0.000101,20.000000,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
1,2,content_11bf4c33adea7bdc,MONITOR,NO_CTR_OPPORTUNITY,331832,0.000000,0.0,0.000000,19.253976,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
2,3,content_d0acf7062bc6b257,MONITOR,NO_CTR_OPPORTUNITY,292374,0.000000,0.0,0.000000,19.062238,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
3,4,content_c60628276389acbb,MONITOR,NO_CTR_OPPORTUNITY,282446,0.000000,0.0,0.000000,19.009915,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
4,5,content_99fc6465edb0e52c,MONITOR,NO_CTR_OPPORTUNITY,278990,0.000000,0.0,0.000000,18.991268,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
5,6,content_39e19a3ec2d95f9d,MONITOR,NO_CTR_OPPORTUNITY,264399,0.000000,0.0,0.000000,18.909910,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
6,7,content_987d251ee617d9c6,MONITOR,NO_CTR_OPPORTUNITY,261191,0.002998,0.0,0.002998,18.891421,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
7,8,content_32c5cc913fb4ff41,MONITOR,NO_CTR_OPPORTUNITY,256491,0.000008,0.0,0.000008,18.863919,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
8,9,content_012de75c008aa653,MONITOR,NO_CTR_OPPORTUNITY,248191,0.000000,0.0,0.000000,18.814097,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."
9,10,content_7471467133493ce6,MONITOR,NO_CTR_OPPORTUNITY,209105,0.000000,0.0,0.000000,18.554556,Higher confidence: strong relative impression ...,"Query mix, SERP features, position benchmark, ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



The weakest positive-priority picks are the items near the bottom of the review queue.

They are less convincing because their combined scores are lower.

The baseline uses only information available in the scoring window: CTR, position context, and impressions.

Future outcomes, decline labels, product-generated flags, and action labels are not used to calculate the score.

In [22]:
positive_queue = baseline_queue[
    baseline_queue["action_label"]
    == "REVIEW_CTR_OPPORTUNITY"
].copy()

weak_picks = positive_queue.tail(10)

print("Weakest positive-priority picks:")

display(weak_picks)

Weakest positive-priority picks:


,rank,content_hash_id,impressions_90d,ctr_90d,expected_ctr,ctr_gap,ctr_gap_score,volume_score,baseline_score,reason_code,action_label


In [23]:
scoring_inputs = [
    "ctr_90d",
    "expected_ctr",
    "ctr_gap",
    "impressions_90d",
    "avg_position_90d"
]

print("Actual scoring inputs:")

for col in scoring_inputs:
    print("-", col)


forbidden_keywords = [
    "future",
    "label",
    "target",
    "trend",
    "declin",
    "flag",
    "action"
]

violations = []

for col in scoring_inputs:

    for keyword in forbidden_keywords:

        if keyword in col.lower():
            violations.append(col)


if violations:

    raise ValueError(
        f"Potential leakage detected: {violations}"
    )

print("\nPASS: No obvious leakage fields are used.")

Actual scoring inputs:
- ctr_90d
- expected_ctr
- ctr_gap
- impressions_90d
- avg_position_90d

PASS: No obvious leakage fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.